Задание 1. Ручное локальное выравнивание

In [20]:
!pip install biopython matplotlib numpy -q

import numpy as np
from Bio import SeqIO, Entrez
import matplotlib.pyplot as plt
import urllib.request
import os


In [26]:
seq3_1 = "ATGCAGCAGCAGCCA"
seq3_2 = "ATATAT"

def needleman_wunsch_affine(seq1, seq2, match=3, mismatch=-3, gap_open=-10, gap_extend=-1):
    n, m = len(seq1), len(seq2)

    M  = np.full((n+1, m+1), -np.inf)
    Ix = np.full((n+1, m+1), -np.inf)
    Iy = np.full((n+1, m+1), -np.inf)
    M[0,0] = 0

    for i in range(1, n+1):
        Ix[i,0] = gap_open + i * gap_extend
        M[i,0]  = Ix[i,0]
    for j in range(1, m+1):
        Iy[0,j] = gap_open + j * gap_extend
        M[0,j]  = Iy[0,j]
    for i in range(1, n+1):
        for j in range(1, m+1):
            # M[i][j]
            match_score = match if seq1[i-1]==seq2[j-1] else mismatch
            M[i,j] = max(M[i-1,j-1], Ix[i-1,j-1], Iy[i-1,j-1]) + match_score
            # Ix[i][j] (гэп в seq1)
            Ix[i,j] = max(M[i-1,j] + gap_open + gap_extend,
                          Ix[i-1,j] + gap_extend)
            # Iy[i][j] (гэп в seq2)
            Iy[i,j] = max(M[i,j-1] + gap_open + gap_extend,
                          Iy[i,j-1] + gap_extend)
    score = max(M[n,m], Ix[n,m], Iy[n,m])

    al1, al2 = [], []
    i, j = n, m

    state = 'M'
    if Ix[n,m] == score: state = 'Ix'
    elif Iy[n,m] == score: state = 'Iy'
    while i > 0 or j > 0:
        if state == 'M':
            match_score = match if seq1[i-1]==seq2[j-1] else mismatch
            if M[i,j] == M[i-1,j-1] + match_score:
                al1.append(seq1[i-1]); al2.append(seq2[j-1])
                i -= 1; j -= 1; state = 'M'
            elif M[i,j] == Ix[i-1,j-1] + match_score:
                al1.append(seq1[i-1]); al2.append(seq2[j-1])
                i -= 1; j -= 1; state = 'Ix'
            elif M[i,j] == Iy[i-1,j-1] + match_score:
                al1.append(seq1[i-1]); al2.append(seq2[j-1])
                i -= 1; j -= 1; state = 'Iy'
        elif state == 'Ix':
            if Ix[i,j] == M[i-1,j] + gap_open + gap_extend:
                al1.append(seq1[i-1]); al2.append('-')
                i -= 1; state = 'M'
            elif Ix[i,j] == Ix[i-1,j] + gap_extend:
                al1.append(seq1[i-1]); al2.append('-')
                i -= 1; state = 'Ix'
        elif state == 'Iy':
            if Iy[i,j] == M[i,j-1] + gap_open + gap_extend:
                al1.append('-'); al2.append(seq2[j-1])
                j -= 1; state = 'M'
            elif Iy[i,j] == Iy[i,j-1] + gap_extend:
                al1.append('-'); al2.append(seq2[j-1])
                j -= 1; state = 'Iy'
    al1.reverse(); al2.reverse()
    return score, ''.join(al1), ''.join(al2)

# Линейный штраф (нужно match=3, mismatch=-3)
F_lin, score_lin, al1_lin, al2_lin = needleman_wunsch(seq3_1, seq3_2, match=3, mismatch=-3, gap=-4)
print("Линейный штраф (gap = -4):")
print(f"Score: {score_lin}")
print(al1_lin)
print(al2_lin)

# Аффинный штраф
score_aff, al1_aff, al2_aff = needleman_wunsch_affine(seq3_1, seq3_2, match=3, mismatch=-3,
                                                      gap_open=-10, gap_extend=-1)
print("\nАффинный штраф (open = -10, extend = -1):")
print(f"Score: {score_aff}")
print(al1_aff)
print(al2_aff)
print("\nБиологическое обоснование: Аффинная модель лучше описывает реальный процесс вставок или делеций, "
      "где инициация разрыв энергетически более затратен, чем его удлинение.")

Линейный штраф (gap = -4):
Score: -30
ATGCAGCAGCAGCCA
AT-----A-TA---T

Аффинный штраф (open = -10, extend = -1):
Score: -19.0
ATGCAGCAGCAGCCA
ATATAT---------

Биологическое обоснование: Аффинная модель лучше описывает реальный процесс вставок или делеций, где инициация разрыв энергетически более затратен, чем его удлинение.
